## Helper functions

In [ ]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler
from src.core.fingerprints import Fingerprints
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

THR = 0.99

def visualize_target(df: pd.DataFrame, target_name: str, n_bins: int = 50) -> None:
    fig, (ax_box, ax_hist) = plt.subplots(
        2, sharex=True, gridspec_kw={"height_ratios": (.15, .85)}, figsize=(10, 8)
    )
    sns.boxplot(x=df[target_name], ax=ax_box, orient='h')
    ax_box.set(xlabel='')
    sns.histplot(df[target_name], bins=n_bins, kde=True, ax=ax_hist)

    ax_box.set_title(f'Distribution of {target_name}')
    plt.show()

def print_stats(df: pd.DataFrame, target_name: str) -> None:
    print(f"Mean: {df[target_name].mean():.4f}")
    print(f"Median: {df[target_name].median():.4f}")
    print(f"Standard Deviation: {df[target_name].std():.4f}")
    print(f"Minimum: {df[target_name].min():.4f}")
    print(f"Maximum: {df[target_name].max():.4f}")

def ecfp_features(smiles_name: str, target_name: str, df: pd.DataFrame, threshold: float = 0.99) -> pd.DataFrame:
    names = ['ecfp']
    params = {
        'ecfp': {'radius': 2, 'size': 1024, 'count': True},
    }
    fingerprints, f_names = Fingerprints().apply(
        smiles=df[smiles_name].tolist(),
        names=names,
        **params
    )
    df_fingerprints = pd.DataFrame(fingerprints, columns=f_names)
    scaler = MinMaxScaler()
    df_features_scaled = scaler.fit_transform(df_fingerprints)
    selector = VarianceThreshold(threshold=1 - threshold)
    selector.fit(df_features_scaled)
    df_fingerprints = df_fingerprints.iloc[:, selector.get_support(indices=True)]
    df_fingerprints[target_name] = df[target_name].values
    df_fingerprints[smiles_name] = df[smiles_name].values
    return df_fingerprints

def check_nans(df: pd.DataFrame) -> None:
    nan_counts = df.isna().sum()
    print("NaN counts per column:")
    print(nan_counts)

def remove_nans(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna().reset_index(drop=True)

def remove_incorrect_molecules(df: pd.DataFrame, smiles_col: str) -> pd.DataFrame:
    from rdkit import Chem
    mols = [Chem.MolFromSmiles(smi) for smi in df[smiles_col]]
    valid_indices = [i for i, m in enumerate(mols) if m is not None]
    return df.iloc[valid_indices].reset_index(drop=True)

def canon_smiles(df: pd.DataFrame, smiles_col: str) -> pd.DataFrame:
    from rdkit import Chem
    df[smiles_col] = df[smiles_col].apply(lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x)) if pd.notna(x) else x)
    return df

def remove_duplicates(df: pd.DataFrame, smiles_col: str) -> pd.DataFrame:
    df[smiles_col] = canon_smiles(df, smiles_col)[smiles_col]
    return df.drop_duplicates(subset=[smiles_col]).reset_index(drop=True)

def remove_outliers(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    mean = df[target_col].mean()
    std = df[target_col].std()
    lower_bound = mean - 3 * std
    upper_bound = mean + 3 * std
    return df[(df[target_col] >= lower_bound) & (df[target_col] <= upper_bound)].reset_index(drop=True)


## CNOHF

In [ ]:
df_cnohf = pd.read_excel('../data/raw/cnohf.xlsx')
df_cnohf

In [ ]:
check_nans(df_cnohf)

In [ ]:
df_cnohf = df_cnohf[['SMILES', 'detonation velocity (km/s)']]
df_cnohf.rename(columns={'SMILES': 'smiles', 'detonation velocity (km/s)': 'detonation_velocity'}, inplace=True)
df_cnohf.head()

In [ ]:
df_cnohf = remove_nans(df_cnohf)
df_cnohf = remove_duplicates(df_cnohf, 'smiles')
df_cnohf = remove_incorrect_molecules(df_cnohf, 'smiles')
len(df_cnohf)

In [ ]:
visualize_target(df_cnohf, 'detonation_velocity')
print_stats(df_cnohf, 'detonation_velocity')

In [ ]:
df_cnohf = remove_outliers(df_cnohf, 'detonation_velocity')
len(df_cnohf)

In [ ]:
visualize_target(df_cnohf, 'detonation_velocity')
print_stats(df_cnohf, 'detonation_velocity')

In [ ]:
df_cnohf_ecfp = ecfp_features('smiles', 'detonation_velocity', df_cnohf, threshold=THR)
df_cnohf_ecfp.head()

In [ ]:
(df_cnohf_ecfp['detonation_velocity'].max() - df_cnohf_ecfp['detonation_velocity'].min()) * 1/10

In [ ]:
df_cnohf_ecfp.to_csv('../data/cnohf_data/cnohf_ecfp.csv', index=False)

## Photoswitches

In [ ]:
df_photo = pd.read_csv('../data/raw/photoswitches.csv')
df_photo

In [ ]:
check_nans(df_photo)

In [ ]:
df_photo = df_photo[['SMILES', 'E isomer pi-pi* wavelength in nm']]
df_photo.rename(columns={'SMILES': 'smiles', 'E isomer pi-pi* wavelength in nm': 'e_isomer_pi_pi'}, inplace=True)
df_photo.head()

In [ ]:
df_photo = remove_nans(df_photo)
df_photo = remove_duplicates(df_photo, 'smiles')
df_photo = remove_incorrect_molecules(df_photo, 'smiles')
len(df_photo)

In [ ]:
visualize_target(df_photo, 'e_isomer_pi_pi')
print_stats(df_photo, 'e_isomer_pi_pi')

In [ ]:
df_photo = remove_outliers(df_photo, 'e_isomer_pi_pi')
len(df_photo)

In [ ]:
visualize_target(df_photo, 'e_isomer_pi_pi')
print_stats(df_photo, 'e_isomer_pi_pi')

In [ ]:
df_photo_ecfp = ecfp_features('smiles', 'e_isomer_pi_pi', df_photo, threshold=THR)
df_photo_ecfp

In [ ]:
(df_photo_ecfp['e_isomer_pi_pi'].max() - df_photo_ecfp['e_isomer_pi_pi'].min()) * 1/10

In [ ]:
df_photo_ecfp.to_csv('../data/photoswitch_data/photoswitch_ecfp.csv', index=False)

## Polymers

In [ ]:
df_poly = pd.read_csv('../data/raw/polymers.csv')
df_poly

In [ ]:
check_nans(df_poly)

In [ ]:
df_poly = df_poly[['SMILES', 'Tg/K']]
df_poly.rename(columns={'SMILES': 'smiles', 'Tg/K': 'Tg'}, inplace=True)
df_poly.head()

In [ ]:
df_poly = remove_nans(df_poly)
df_poly = remove_incorrect_molecules(df_poly, 'smiles')
df_poly = remove_duplicates(df_poly, 'smiles')
len(df_poly)

In [ ]:
visualize_target(df_poly, 'Tg')
print_stats(df_poly, 'Tg')

In [ ]:
df_poly = remove_outliers(df_poly, 'Tg')
len(df_poly)

In [ ]:
visualize_target(df_poly, 'Tg')
print_stats(df_poly, 'Tg')

In [ ]:
df_poly_ecfp = ecfp_features('smiles', 'Tg', df_poly, threshold=THR)
df_poly_ecfp.head()

In [ ]:
(df_poly_ecfp['Tg'].max() - df_poly_ecfp['Tg'].min()) * 1/10

In [ ]:
df_poly_ecfp.to_csv('../data/polymers_data/polymers_ecfp.csv', index=False)

## Redox

In [ ]:
redox_smiles = pd.read_csv('../data/raw/redox_smiles.csv')
redox_target = pd.read_csv('../data/raw/redox_pbe0_dG.csv')
df_redox = redox_smiles.merge(redox_target, on='runs')
df_redox

In [ ]:
check_nans(df_redox)

In [ ]:
df_redox = df_redox[['smiles', 'dGox']]
df_redox.head()

In [ ]:
df_redox = remove_nans(df_redox)
df_redox = remove_duplicates(df_redox, 'smiles')
df_redox = remove_incorrect_molecules(df_redox, 'smiles')
len(df_redox)

In [ ]:
visualize_target(df_redox, 'dGox')
print_stats(df_redox, 'dGox')

In [ ]:
df_redox = remove_outliers(df_redox, 'dGox')
len(df_redox)

In [ ]:
visualize_target(df_redox, 'dGox')
print_stats(df_redox, 'dGox')

In [ ]:
df_redox_ecfp = ecfp_features('smiles', 'dGox', df_redox, threshold=THR)
df_redox_ecfp.head()

In [ ]:
(df_redox_ecfp['dGox'].max() - df_redox_ecfp['dGox'].min()) * 1/10

In [ ]:
df_redox_ecfp.to_csv('../data/redox_data/redox_ecfp.csv', index=False)

## COF dataset

In [ ]:
df_cof = pd.read_csv('../data/raw/cof.csv')
df_cof

In [ ]:
check_nans(df_cof)

In [ ]:
df_cof = remove_nans(df_cof)
df_cof = remove_duplicates(df_cof, 'smiles')
df_cof = remove_incorrect_molecules(df_cof, 'smiles')
len(df_cof)

In [ ]:
visualize_target(df_cof, 'capacity_max')
print_stats(df_cof, 'capacity_max')

In [ ]:
df_cof = remove_outliers(df_cof, 'capacity_max')
len(df_cof)

In [ ]:
visualize_target(df_cof, 'capacity_max')
print_stats(df_cof, 'capacity_max')

In [ ]:
def ecfp_descriptor_features(smiles_name: str, target_name: str, df: pd.DataFrame, threshold: float = 0.99) -> pd.DataFrame:
    names = ['ecfp', 'descriptor']
    params = {
        'ecfp': {'radius': 2, 'size': 1024, 'count': True},
        'descriptor': {}
    }
    fingerprints, f_names = Fingerprints().apply(
        smiles=df_cof['smiles'].tolist(),
        names=names,
        **params
    )
    df_fingerprints = pd.DataFrame(fingerprints, columns=f_names)
    scaler = MinMaxScaler()
    df_features_scaled = scaler.fit_transform(df_fingerprints)
    selector = VarianceThreshold(threshold=1 - threshold)
    selector.fit(df_features_scaled)
    df_fingerprints = df_fingerprints.iloc[:, selector.get_support(indices=True)]
    df_fingerprints[target_name] = df[target_name].values
    df_fingerprints[smiles_name] = df[smiles_name].values
    return df_fingerprints

df_cof_ecfp_desc = ecfp_descriptor_features('smiles', 'capacity_max', df_cof, threshold=THR)
df_cof_ecfp_desc.head()

In [ ]:
(df_cof_ecfp_desc['capacity_max'].max() - df_cof_ecfp_desc['capacity_max'].min()) * 1 / 10

In [ ]:
df_cof_ecfp_desc.to_csv('../data/cof_data/cof_ecfp_descriptor.csv', index=False)